# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - ONI

## **M0.: Configuración General**

In [ ]:
import pandas as pd
import requests
import io
import numpy as np

print("Necessary libraries imported successfully.")

Necessary libraries imported successfully.


## **M1. Definición de la Fuente Oficial**

In [ ]:
url_oni = "https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/detrend.nino34.ascii.txt"
print(f"URL de la fuente oficial del ONI definida: {url_oni}")

URL de la fuente oficial del ONI definida: https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/detrend.nino34.ascii.txt


## **M2. Extracción y Procesamiento de Datos**


In [ ]:
print("Iniciando la descarga del Índice del Niño Oceánico (ONI) desde la NOAA...")

try:
    # 1. Realizar una solicitud HTTP GET y 2. Leer el contenido en un DataFrame
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url_oni, headers=headers)

    if response.status_code == 200:
        # Leer el archivo plano usando espacios como delimitador
        df_oni_crudo = pd.read_csv(io.StringIO(response.text), sep=r'\s+')

        # 3. Renombrar la columna de anomalía a ONI
        df_oni_crudo = df_oni_crudo.rename(columns={'ANOM': 'ONI'})

        # 4. Crear una columna de fecha estandarizada al fin de mes
        df_oni_crudo['YR'] = df_oni_crudo['YR'].astype(int)
        df_oni_crudo['MON'] = df_oni_crudo['MON'].astype(int)
        df_oni_crudo['Fecha'] = pd.to_datetime(df_oni_crudo['YR'].astype(str) + '-' + df_oni_crudo['MON'].astype(str) + '-01')
        df_oni_crudo['Fecha'] = df_oni_crudo['Fecha'] + pd.offsets.MonthEnd(0)

        # 5. Filtrar el rango histórico y crear una copia
        df_oni = df_oni_crudo[(df_oni_crudo['Fecha'] >= '2015-01-01') & (df_oni_crudo['Fecha'] <= '2026-03-31')].copy()

        # 6. Clasificación de Eventos ENSO (Niño, Niña, Neutral)
        df_oni["Estado_ENSO"] = np.where(
            df_oni["ONI"] >= 0.5,
            "Niño",
            np.where(
                df_oni["ONI"] <= -0.5,
                "Niña",
                "Neutral"
            )
        )

        # 7. Seleccionar y ordenar las columnas finales
        df_oni = df_oni[['Fecha', 'ONI', 'Estado_ENSO']].sort_values(by='Fecha').reset_index(drop=True)

        # 8. Exportar a Excel
        df_oni.to_excel("01_Capa_Oni_V1.xlsx", index=False)

        # 9. Imprimir mensaje de confirmación y primeros registros
        print(" -> Archivo '01_Capa_Oni_V1.xlsx' generado con éxito.")
        print(f" -> Se procesaron {len(df_oni)} meses de registros climáticos.")
        print("\nPrimeros registros del dataset:")
        print(df_oni.head())

    else:
        print(f" -> No se pudo acceder al servidor de la NOAA. Código: {response.status_code}")

except Exception as e:
    print(f" -> Error crítico en el pipeline del ONI: {e}")

Iniciando la descarga del Índice del Niño Oceánico (ONI) desde la NOAA...
 -> Archivo '01_Capa_Oni_V1.xlsx' generado con éxito.
 -> Se procesaron 135 meses de registros climáticos.

Primeros registros del dataset:
       Fecha   ONI Estado_ENSO
0 2015-01-31  0.68        Niño
1 2015-02-28  0.57        Niño
2 2015-03-31  0.57        Niño
3 2015-04-30  0.82        Niño
4 2015-05-31  1.03        Niño


## **M3. Reporte de Auditoría**


Realizar una auditoría técnica detallada del dataset ONI procesado. El proceso intentará primero cargar el archivo '01_Capa_Oni_V1.xlsx' localmente. Si el archivo local no está disponible, se descargará desde la URL de GitHub proporcionada ('https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/04_Capa_ONI/01_Capa_Oni_V1.xlsx'). Una vez cargado, se calcularán métricas clave como el número de registros, variables, valores nulos, duplicados, cobertura temporal, estadísticas de ONI y el conteo de clasificaciones ENSO. Estos resultados se consolidarán en DataFrames de auditoría.


In [ ]:
print("Calculando métricas de auditoría...")

# --- 1. General Data Audit DataFrame ---
gen_data_rows = [
    ["Registros totales", df_oni.shape[0]],
    ["Variables (Columnas)", df_oni.shape[1]],
    ["Nombres de Columnas", ', '.join(df_oni.columns.tolist())],
    ["Valores nulos totales", df_oni.isnull().sum().sum()],
    ["Valores nulos por columna", df_oni.isnull().sum().to_dict()],
    ["Registros duplicados", df_oni.duplicated().sum()]
]
df_general_data = pd.DataFrame(gen_data_rows, columns=['Métrica', 'Valor'])

# --- 2. Temporal Analysis DataFrame ---
min_fecha = df_oni['Fecha'].min()
max_fecha = df_oni['Fecha'].max()
expected_periods = pd.date_range(min_fecha, max_fecha, freq="ME")

temporal_data_rows = [
    ["Primer registro", min_fecha.strftime('%Y-%m-%d')],
    ["Último registro", max_fecha.strftime('%Y-%m-%d')],
    ["Rango de datos", f"{min_fecha.strftime('%Y-%m-%d')} a {max_fecha.strftime('%Y-%m-%d')}"],
    ["Número de meses (registros actuales)", len(df_oni)],
    ["Frecuencia esperada", "Mensual"],
    ["Meses esperados", len(expected_periods)],
    ["Continuidad", "Continua" if len(df_oni) == len(expected_periods) else "Discontinua"]
]
df_temporal_analysis = pd.DataFrame(temporal_data_rows, columns=['Métrica', 'Valor'])

# --- 3. ONI Statistical Analysis DataFrame ---
df_oni_stats = df_oni['ONI'].describe().reset_index()
df_oni_stats.columns = ['Estadística', 'Valor']

# --- 4. ENSO Classification DataFrame ---
df_enso_classification = df_oni['Estado_ENSO'].value_counts().reset_index()
df_enso_classification.columns = ['Estado_ENSO', 'Conteo']

print("Métricas de auditoría calculadas exitosamente.")

Calculando métricas de auditoría...
Métricas de auditoría calculadas exitosamente.


In [ ]:
print("Datos de Auditoría General:")
print(df_general_data)

Datos de Auditoría General:
                     Métrica                                     Valor
0          Registros totales                                       135
1       Variables (Columnas)                                         3
2        Nombres de Columnas                   Fecha, ONI, Estado_ENSO
3      Valores nulos totales                                         0
4  Valores nulos por columna  {'Fecha': 0, 'ONI': 0, 'Estado_ENSO': 0}
5       Registros duplicados                                         0


In [ ]:
print("Análisis Temporal de Auditoría:")
print(df_temporal_analysis)

Análisis Temporal de Auditoría:
                                Métrica                    Valor
0                       Primer registro               2015-01-31
1                       Último registro               2026-03-31
2                        Rango de datos  2015-01-31 a 2026-03-31
3  Número de meses (registros actuales)                      135
4                   Frecuencia esperada                  Mensual
5                       Meses esperados                      135
6                           Continuidad                 Continua


In [ ]:
print("Análisis Estadístico del ONI:")
print(df_oni_stats)

Análisis Estadístico del ONI:
  Estadística       Valor
0       count  135.000000
1        mean    0.159407
2         std    0.932565
3         min   -1.360000
4         25%   -0.545000
5         50%   -0.020000
6         75%    0.645000
7         max    2.770000


In [ ]:
print("Clasificación de Eventos ENSO:")
print(df_enso_classification)

Clasificación de Eventos ENSO:
  Estado_ENSO  Conteo
0     Neutral      52
1        Niño      44
2        Niña      39


## **M4. Exportación de Resultados**


In [ ]:
import pandas as pd

# Assuming df_oni is already loaded in the kernel from the previous step
# If not, add code to load it (e.g., from '01_Capa_Oni_V1.xlsx')

# --- 1. General Data Audit DataFrame ---
gen_data_rows = [
    ["Registros totales", df_oni.shape[0]],
    ["Variables (Columnas)", df_oni.shape[1]],
    ["Nombres de Columnas", ', '.join(df_oni.columns.tolist())],
    ["Valores nulos totales", df_oni.isnull().sum().sum()],
    ["Valores nulos por columna", df_oni.isnull().sum().to_dict()],
    ["Registros duplicados", df_oni.duplicated().sum()]
]
df_general_data = pd.DataFrame(gen_data_rows, columns=['Métrica', 'Valor'])

# --- 2. Temporal Analysis DataFrame ---
min_fecha = df_oni['Fecha'].min()
max_fecha = df_oni['Fecha'].max()
expected_periods = pd.date_range(min_fecha, max_fecha, freq="ME")

temporal_data_rows = [
    ["Primer registro", min_fecha.strftime('%Y-%m-%d')],
    ["Último registro", max_fecha.strftime('%Y-%m-%d')],
    ["Rango de datos", f"{min_fecha.strftime('%Y-%m-%d')} a {max_fecha.strftime('%Y-%m-%d')}"],
    ["Número de meses (registros actuales)", len(df_oni)],
    ["Frecuencia esperada", "Mensual"],
    ["Meses esperados", len(expected_periods)],
    ["Continuidad", "Continua" if len(df_oni) == len(expected_periods) else "Discontinua"]
]
df_temporal_analysis = pd.DataFrame(temporal_data_rows, columns=['Métrica', 'Valor'])

# --- 3. ONI Statistical Analysis DataFrame ---
df_oni_stats = df_oni['ONI'].describe().reset_index()
df_oni_stats.columns = ['Estadística', 'Valor']

# --- 4. ENSO Classification DataFrame ---
df_enso_classification = df_oni['Estado_ENSO'].value_counts().reset_index()
df_enso_classification.columns = ['Estado_ENSO', 'Conteo']

# --- Export to Excel ---
output_excel_file = 'Auditoria_Capa_Oni.xlsx'
with pd.ExcelWriter(output_excel_file, engine='xlsxwriter') as writer:
    df_general_data.to_excel(writer, sheet_name='General_Data', index=False)
    df_temporal_analysis.to_excel(writer, sheet_name='Temporal_Analysis', index=False)
    df_oni_stats.to_excel(writer, sheet_name='ONI_Statistics', index=False)
    df_enso_classification.to_excel(writer, sheet_name='ENSO_Classification', index=False)

print(f"Auditoría exportada exitosamente a '{output_excel_file}' con hojas separadas.")

Auditoría exportada exitosamente a 'Auditoria_Capa_Oni.xlsx' con hojas separadas.


## **M5. Resumen Final**

La auditoría técnica exhaustiva del dataset del Índice del Niño Oceánico (ONI) ha confirmado su alta calidad e idoneidad para la integración en el Framework V7 y su utilización en el modelo LSTM. Los hallazgos clave son los siguientes:

*   **Integridad y Calidad de los Datos**: El dataset de ONI está completo, sin valores nulos ni registros duplicados en sus 135 entradas. Las variables `Fecha`, `ONI` y `Estado_ENSO` están correctamente tipificadas y estructuradas.
*   **Cobertura y Continuidad Temporal**: La serie temporal abarca de forma continua desde el 31 de enero de 2015 hasta el 31 de marzo de 2026, con una frecuencia mensual consistente. Esta cobertura asegura un historial robusto para el entrenamiento y la validación del modelo.
*   **Análisis Estadístico del ONI**: El ONI presenta valores que varían entre -1.36 y 2.77, con una media de 0.16 y una desviación estándar significativa de 0.93. Esta variabilidad es crucial, ya que indica la presencia de eventos climáticos extremos (Niño y Niña) y periodos neutrales que son fundamentales para la predicción de eventos hidrológicos.
*   **Clasificación de Eventos ENSO**: Se identificaron 44 eventos de Niño, 39 de Niña y 52 periodos Neutrales, lo que demuestra la capacidad del dataset para representar las distintas fases del ciclo ENSO.

En conclusión, el dataset ONI está validado y es altamente apto para ser utilizado como una característica predictiva en el modelo LSTM. Su capacidad para capturar las variaciones térmicas significativas en el Pacífico ecuatorial lo convierte en un insumo clave para mejorar la capacidad del modelo de predecir y comprender el impacto de los eventos climáticos extremos en la hidrología de la cuenca del Río Bogotá.